In [ ]:
# --- Importaciones ---------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    StratifiedKFold, GridSearchCV, RandomizedSearchCV,
    validation_curve, learning_curve,
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
import scipy.stats as stats

import sklearn
import scipy
print(f'numpy:   {np.__version__}')   # anota la versión
print(f'pandas:  {pd.__version__}')
print(f'sklearn: {sklearn.__version__}')
print(f'scipy:   {scipy.__version__}')

In [ ]:
# --- Dataset California Housing (regresión) -------------------------
housing = fetch_california_housing(as_frame=True)
X_housing = housing.data
y_housing = housing.target

semilla = 42
X_tr, X_te, y_tr, y_te = train_test_split(
    X_housing, y_housing,
    test_size=0.2, random_state=semilla,
)
print(f'Train: {X_tr.shape} | Test: {X_te.shape}')

In [ ]:
# --- Evaluación con validación cruzada ------------------------------
modelo_rf = RandomForestRegressor(
    n_estimators=100, random_state=semilla, n_jobs=-1
)

scores = cross_val_score(
    modelo_rf, X_tr, y_tr,
    cv=5,
    scoring='r2',
    n_jobs=-1,
)

print(f'R² por fold: {scores.round(4)}')
print(f'Media: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# --- cross_validate: varias métricas simultáneamente ----------------
resultados = cross_validate(
    modelo_rf, X_tr, y_tr,
    cv=5,
    scoring=['r2', 'neg_mean_absolute_error'],
    return_train_score=True,
    n_jobs=-1,
)

r2_val   = resultados['test_r2'].mean()
r2_train = resultados['train_r2'].mean()
mae_val  = -resultados['test_neg_mean_absolute_error'].mean()

print(f'R² train: {r2_train:.4f} | R² val: {r2_val:.4f}')
print(f'MAE validación: {mae_val:.4f}')

In [ ]:
# --- Cargar Titanic para clasificación ------------------------------
import seaborn as sns

try:
    df = sns.load_dataset('titanic')
except Exception as e:
    print(f'Error al cargar Titanic ({e}). Usando URL alternativa.')
    url = ('https://raw.githubusercontent.com/datasciencedojo/'
        'datasets/master/titanic.csv')
    df = pd.read_csv(url)

cols_num_tit = ['age', 'fare', 'sibsp', 'parch']
cols_cat_tit = ['sex', 'embarked', 'class']
X_tit = df[cols_num_tit + cols_cat_tit]
y_tit = df['survived']
# Nota: no separamos un test set aquí porque el objetivo es demostrar
# StratifiedKFold. En producción siempre reservarías un test final.
# --- Pipeline con preprocesamiento correcto para datos nominales ----
from sklearn.linear_model import LogisticRegression

# Variables nominales (sex, embarked, class) requieren OneHotEncoder:
# no tienen orden inherente, OrdinalEncoder sería incorrecto aquí.
prep_tit = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), cols_num_tit),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore',
                            sparse_output=False)),
    ]), cols_cat_tit),
])

lr = LogisticRegression(max_iter=1000, random_state=semilla)
pipe_lr = Pipeline([
    ('prep', prep_tit),
    ('clf', lr),
])

# --- StratifiedKFold explícito --------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)

scores_skf = cross_val_score(
    pipe_lr, X_tit, y_tit, cv=skf, scoring='accuracy'
)
print(f'Accuracy: {scores_skf.mean():.4f} ± {scores_skf.std():.4f}')

In [ ]:
# --- GridSearchCV sobre RandomForestRegressor -----------------------
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 10, 20],
    'max_features': ['sqrt', 'log2'],
}
# 3 x 3 x 2 = 18 combinaciones x 5 folds = 90 entrenamientos

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=semilla, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    refit=True,       # refit sobre todo X_tr con los mejores params
    verbose=1,
)
grid_search.fit(X_tr, y_tr)

In [ ]:
# --- Mejores hiperparámetros y score de validación ------------------
print(f'Mejores params: {grid_search.best_params_}')
print(f'Mejor CV score: {grid_search.best_score_:.4f}')
# Resultados típicos:
# Mejores params: {'max_depth': None, 'max_features': 'sqrt',
#                  'n_estimators': 200}
# Mejor CV score: 0.8103

# Score final en test (solo se hace UNA vez, al final)
mejor_modelo = grid_search.best_estimator_
r2_test = r2_score(y_te, mejor_modelo.predict(X_te))
print(f'R² en test: {r2_test:.4f}')

# --- Explorar todos los resultados como DataFrame -------------------
resultados_grid = pd.DataFrame(grid_search.cv_results_)
cols_utiles = [
    'param_n_estimators', 'param_max_depth',
    'param_max_features', 'mean_test_score', 'std_test_score',
]
print(resultados_grid[cols_utiles].sort_values(
    'mean_test_score', ascending=False
).head(5).to_string(index=False))

In [ ]:
# --- RandomizedSearchCV: muestreo de distribuciones -----------------
param_dist = {
    'n_estimators':    stats.randint(50, 500),
    'max_depth':       [None, 5, 10, 15, 20, 30],
    'max_features':    ['sqrt', 'log2', 0.3, 0.5],
    'min_samples_leaf': stats.randint(1, 10),
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=semilla, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=30,        # solo 30 combinaciones de las miles posibles
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=semilla,
    refit=True,
    verbose=1,
)
random_search.fit(X_tr, y_tr)

print(f'Mejores params: {random_search.best_params_}')
print(f'Mejor CV score: {random_search.best_score_:.4f}')
r2_rand = r2_score(y_te, random_search.best_estimator_.predict(X_te))
print(f'R² en test: {r2_rand:.4f}')

In [ ]:
# --- Pipeline Titanic + GridSearchCV --------------------------------
cols_num = ['age', 'fare', 'sibsp', 'parch']
cols_cat = ['sex', 'embarked', 'class']

preprocesador = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), cols_num),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OneHotEncoder(handle_unknown='ignore',
                            sparse_output=False)),
    ]), cols_cat),
])

pipeline_tit = Pipeline([
    ('prep', preprocesador),
    ('clf', RandomForestClassifier(
        random_state=semilla, n_jobs=-1
    )),
])

# Notación doble guion bajo: paso__parámetro
param_grid_pipe = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth':    [None, 10],
    'prep__num__strategy': ['mean', 'median'],
}

grid_pipe = GridSearchCV(
    pipeline_tit,
    param_grid_pipe,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
)
grid_pipe.fit(X_tit, y_tit)

print(f'Mejores params: {grid_pipe.best_params_}')
print(f'Mejor CV accuracy: {grid_pipe.best_score_:.4f}')

In [ ]:
# --- Curva de validación: efecto de max_depth -----------------------
profundidades = [1, 2, 3, 5, 8, 12, 20, None]

scores_train, scores_val = validation_curve(
    RandomForestRegressor(n_estimators=50,
                        random_state=semilla, n_jobs=-1),
    X_tr, y_tr,
    param_name='max_depth',
    param_range=profundidades,
    cv=5,
    scoring='r2',
    n_jobs=-1,
)

media_tr  = scores_train.mean(axis=1)
media_val = scores_val.mean(axis=1)
etiquetas = [str(p) if p is not None else 'None' for p in profundidades]

plt.figure(figsize=(8, 4))
plt.plot(etiquetas, media_tr,  'o-', label='Train')
plt.plot(etiquetas, media_val, 's--', label='Validación CV')
plt.xlabel('max_depth')
plt.ylabel('R²')
plt.title('Curva de validación — max_depth en Random Forest')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Curva de aprendizaje -------------------------------------------
tamaños = np.linspace(0.1, 1.0, 8)

sizes_tr, scores_tr, scores_val = learning_curve(
    RandomForestRegressor(n_estimators=100,
                        random_state=semilla, n_jobs=-1),
    X_tr, y_tr,
    train_sizes=tamaños,
    cv=5,
    scoring='r2',
    n_jobs=-1,
)

plt.figure(figsize=(8, 4))
plt.plot(sizes_tr, scores_tr.mean(axis=1),  'o-', label='Train')
plt.plot(sizes_tr, scores_val.mean(axis=1), 's--', label='Validación CV')
plt.xlabel('Tamaño del conjunto de entrenamiento')
plt.ylabel('R²')
plt.title('Curva de aprendizaje — Random Forest')
plt.legend()
plt.tight_layout()
plt.show()